In [1]:
import subprocess, sys, os, shutil, glob, re
from datetime import datetime

# ═══════════════════════════════════════════════════════════════
# PATHS
# ═══════════════════════════════════════════════════════════════
DATA_BASE  = "/kaggle/input/datasets/ajfaisal002/lomix-synapse-preprocessed"
TRAIN_DATA = f"{DATA_BASE}/synapse/synapse/train_npz_new"
TEST_DATA  = f"{DATA_BASE}/synapse/synapse/test_vol_h5_new"
LIST_DIR   = f"{DATA_BASE}/synapse/synapse/lists/lists_Synapse"
BUNDLE_DIR = f"{DATA_BASE}/lomix_restart_bundle"

# ══════════════════════════════════════════════════════════════
# ▼▼▼  ONLY CHANGE THESE TWO LINES EACH SESSION  ▼▼▼
# ══════════════════════════════════════════════════════════════
CHUNKS      = [(30, 150)]
RESUME_PATH = "/kaggle/input/datasets/ajfaisal002/lomix-synapse-preprocessed/lomix_ckpt_epoch30/last.pth"
# ══════════════════════════════════════════════════════════════
# ▲▲▲  ONLY CHANGE THESE TWO LINES EACH SESSION  ▲▲▲
# ══════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════
# STEP 1 — Clone repo
# ═══════════════════════════════════════════════════════════════
print("── Step 1: Cloning repo ──────────────────────────────────")
subprocess.run(
    ["git", "clone", "https://github.com/SLDGroup/LoMix.git",
     "/kaggle/working/LoMix"],
    capture_output=True
)
os.chdir("/kaggle/working/LoMix")
print("✓ Repo cloned")

# ═══════════════════════════════════════════════════════════════
# STEP 2 — Install packages
# ═══════════════════════════════════════════════════════════════
print("\n── Step 2: Installing packages ───────────────────────────")
pkgs = [
    "tensorboardX", "timm==0.6.13", "einops", "medpy", "SimpleITK",
    "scipy", "thop", "ptflops", "segmentation-mask-overlay",
    "ml-collections", "warmup-scheduler", "torchmetrics",
    "albumentations", "loguru", "tabulate", "tifffile", "nibabel"
]
subprocess.run(
    [sys.executable, "-m", "pip", "install"] + pkgs + ["-q"],
    capture_output=True
)
print("✓ Packages installed")

# ═══════════════════════════════════════════════════════════════
# STEP 3 — Restore patched files + PVT weight from bundle
# ═══════════════════════════════════════════════════════════════
print("\n── Step 3: Restoring files from bundle ───────────────────")

bundle_zip        = glob.glob(f"{BUNDLE_DIR}/*.zip")
bundle_files_direct = (glob.glob(f"{BUNDLE_DIR}/*.py") +
                        glob.glob(f"{BUNDLE_DIR}/*.pth"))

if bundle_zip:
    subprocess.run(["unzip", "-o", bundle_zip[0],
                    "-d", "/kaggle/working/patch"], capture_output=True)
    patch_dir = "/kaggle/working/patch"
    print(f"✓ Unzipped bundle from {bundle_zip[0]}")
elif bundle_files_direct:
    patch_dir = BUNDLE_DIR
    print(f"✓ Using bundle files directly from {BUNDLE_DIR}")
else:
    all_zips = glob.glob(f"{BUNDLE_DIR}/**/*.zip", recursive=True)
    if all_zips:
        subprocess.run(["unzip", "-o", all_zips[0],
                        "-d", "/kaggle/working/patch"], capture_output=True)
        patch_dir = "/kaggle/working/patch"
        print(f"✓ Unzipped from {all_zips[0]}")
    else:
        print(f"⚠ Bundle contents: {os.listdir(BUNDLE_DIR)}")
        patch_dir = BUNDLE_DIR

print(f"  Patch dir: {os.listdir(patch_dir)}")

# Restore patched Python files
for fname in ["train_synapse_lomix.py", "trainer.py"]:
    src = os.path.join(patch_dir, fname)
    dst = f"/kaggle/working/LoMix/{fname}"
    if os.path.isfile(src):
        shutil.copy2(src, dst)
        print(f"✓ Restored {fname}")
    else:
        print(f"  ℹ {fname} not in bundle — will patch from scratch below")

# Restore PVT weight
os.makedirs("/kaggle/working/LoMix/pretrained_pth/pvt", exist_ok=True)
pvt_dst = "/kaggle/working/LoMix/pretrained_pth/pvt/pvt_v2_b2.pth"
pvt_src = os.path.join(patch_dir, "pvt_v2_b2.pth")
if os.path.isfile(pvt_src):
    shutil.copy2(pvt_src, pvt_dst)
else:
    shutil.copy2(f"{DATA_BASE}/pvt_models/pvt_v2_b2.pth", pvt_dst)
print(f"✓ PVT weight ready  ({os.path.getsize(pvt_dst)/1e6:.1f} MB)")

# ═══════════════════════════════════════════════════════════════
# STEP 4 — Apply ALL patches (safe: checks before applying)
# ═══════════════════════════════════════════════════════════════
print("\n── Step 4: Applying patches ──────────────────────────────")

# ── train_synapse_lomix.py ─────────────────────────────────────
train_path = "/kaggle/working/LoMix/train_synapse_lomix.py"
with open(train_path) as f:
    content = f.read()

if "PVT_CASCADE" in content:
    content = content.replace(
        "from lib.networks import PVT_CASCADE, EMCADNet",
        "from lib.networks import EMCADNet"
    )
    print("✓ Fixed PVT_CASCADE import")

if "--resume" not in content:
    content = content.replace(
        "parser.add_argument('--seed', type=int,",
        "parser.add_argument('--resume', type=str,\n"
        "                    default='', "
        "help='path to checkpoint to resume from')\n"
        "parser.add_argument('--seed', type=int,"
    )
    print("✓ Added --resume argument")

with open(train_path, "w") as f:
    f.write(content)

# ── trainer.py ────────────────────────────────────────────────
trainer_path = "/kaggle/working/LoMix/trainer.py"
with open(trainer_path) as f:
    content = f.read()

# Resume block (handles BOTH old raw state_dict AND new dict format)
if "start_epoch" not in content:
    content = content.replace(
        "    model.train()\n    ce_loss = CrossEntropyLoss()",
        """    # ── Resume ──────────────────────────────────────────────
    start_epoch = 0
    if args.resume and os.path.isfile(args.resume):
        checkpoint = torch.load(args.resume, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state' in checkpoint:
            model.load_state_dict(checkpoint['model_state'])
            start_epoch = checkpoint['epoch'] + 1
            best_performance = checkpoint.get('best_performance', 0.0)
        else:
            # Old format: raw state_dict (epoch 30 checkpoint)
            model.load_state_dict(checkpoint)
            start_epoch = 30
            best_performance = 0.0
        print(f"✓ Resumed from epoch {start_epoch}")
    # ────────────────────────────────────────────────────────────

    model.train()
    ce_loss = CrossEntropyLoss()"""
    )
    print("✓ Added resume block to trainer.py")
else:
    # Already has start_epoch — but may have old resume format
    # Replace the resume loading block to handle both formats
    old_resume = """    if args.resume and os.path.isfile(args.resume):
        checkpoint = torch.load(args.resume, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state' in checkpoint:
            model.load_state_dict(checkpoint['model_state'])
            start_epoch = checkpoint['epoch'] + 1
            best_performance = checkpoint.get('best_performance', 0.0)
        else:
            # Old format: raw state_dict (epoch 30 checkpoint)
            model.load_state_dict(checkpoint)
            start_epoch = 30
            best_performance = 0.0
        print(f"✓ Resumed from epoch {start_epoch}")"""

    if old_resume not in content:
        # Has start_epoch but not our dual-format handler — patch it
        content = re.sub(
            r"    if args\.resume and os\.path\.isfile\(args\.resume\):.*?"
            r"print\(f.✓ Resumed from epoch \{start_epoch\}.\)",
            old_resume,
            content,
            flags=re.DOTALL
        )
        print("✓ Updated resume block to handle both checkpoint formats")
    else:
        print("✓ Resume block already correct")

# New-format checkpoint save
if "'model_state'" not in content:
    content = content.replace(
        "        save_mode_path = os.path.join(snapshot_path, 'last.pth')\n"
        "        torch.save(model.state_dict(), save_mode_path)",
        "        save_mode_path = os.path.join(snapshot_path, 'last.pth')\n"
        "        torch.save({\n"
        "            'epoch': epoch_num,\n"
        "            'model_state': model.state_dict(),\n"
        "            'best_performance': best_performance,\n"
        "        }, save_mode_path)"
    )
    print("✓ Updated checkpoint save format")

# Epoch iterator
if "range(start_epoch" not in content:
    content = content.replace(
        "    iterator = tqdm(range(max_epoch), ncols=70)",
        "    iterator = tqdm(range(start_epoch, max_epoch), ncols=70)"
    )
    print("✓ Updated epoch iterator")

with open(trainer_path, "w") as f:
    f.write(content)

print("✓ All patches applied")

# ═══════════════════════════════════════════════════════════════
# STEP 5 — Import check
# ═══════════════════════════════════════════════════════════════
print("\n── Step 5: Import check ──────────────────────────────────")
sys.path.insert(0, "/kaggle/working/LoMix")
try:
    from ptflops import get_model_complexity_info;  print("✓ ptflops")
    from utils.utils import powerset;               print("✓ utils.utils")
    from trainer import trainer_synapse;            print("✓ trainer")
    from lib.networks import EMCADNet;              print("✓ EMCADNet")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    raise

# ═══════════════════════════════════════════════════════════════
# STEP 6 — Helper functions
# ═══════════════════════════════════════════════════════════════

def find_last_pth():
    candidates = glob.glob(
        "/kaggle/working/LoMix/model_pth/**/last.pth", recursive=True
    )
    candidates = [c for c in candidates if 'epo5' not in c]
    return max(candidates, key=os.path.getmtime) if candidates else None


def save_checkpoint_zip(chunk_end):
    last_pth  = find_last_pth()
    best_ptns = glob.glob(
        "/kaggle/working/LoMix/model_pth/**/best.pth", recursive=True)
    loss_ptns = glob.glob(
        "/kaggle/working/LoMix/model_pth/**/*best*.pth", recursive=True)
    files = list(set(([last_pth] if last_pth else []) +
                     best_ptns + loss_ptns))
    if not files:
        print("⚠ No checkpoints found to zip")
        return None
    zip_name = (f"lomix_ckpt_epoch{chunk_end}_"
                f"{datetime.now().strftime('%Y%m%d_%H%M')}.zip")
    zip_path = f"/kaggle/working/{zip_name}"
    subprocess.run(["zip", "-j", zip_path] + files, capture_output=True)
    size_mb = os.path.getsize(zip_path) / 1e6
    print("\n" + "═"*60)
    print(f"  ✅  COMPLETE — epochs up to {chunk_end}")
    print(f"  📦  FILE NAME : {zip_name}")
    print(f"      SIZE      : {size_mb:.1f} MB")
    print(f"  ⬇️   DOWNLOAD  : Output tab → {zip_name}")
    print("═"*60 + "\n")
    return zip_path


def train_chunk(start_hint, total_epochs, resume_path=''):
    cmd = [
        sys.executable, "-W", "ignore",
        "train_synapse_lomix.py",
        "--root_path",   TRAIN_DATA,
        "--volume_path", TEST_DATA,
        "--list_dir",    LIST_DIR,
        "--encoder",     "pvt_v2_b2",
        "--supervision", "lomix",
        "--num_classes", "9",
        "--img_size",    "224",
        "--max_epochs",  str(total_epochs),
        "--batch_size",  "12",
        "--n_gpu",       "1",
        "--base_lr",     "0.0001",
        "--seed",        "2222",
    ]
    if resume_path and os.path.isfile(resume_path):
        cmd += ["--resume", resume_path]
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    hrs = (total_epochs - start_hint) * 4.5 / 60
    print(f"\n🚀  epochs {start_hint} → {total_epochs}  (~{hrs:.1f} hrs)\n")
    result = subprocess.run(cmd, capture_output=False, text=True, env=env)
    return result.returncode

# ═══════════════════════════════════════════════════════════════
# STEP 7 — Run training chunks
# ═══════════════════════════════════════════════════════════════
print("\n── Step 7: Training ──────────────────────────────────────")
print(f"   Chunks      : {CHUNKS}")
print(f"   Resume from : {RESUME_PATH or 'fresh start'}")

import torch

current_resume = RESUME_PATH

for chunk_idx, (start, end) in enumerate(CHUNKS):

    # Check if this chunk already done
    if current_resume and os.path.isfile(current_resume):
        ckpt = torch.load(current_resume, map_location='cpu')
        if isinstance(ckpt, dict) and 'epoch' in ckpt:
            done = ckpt['epoch']
            if done >= end - 1:
                print(f"⏭  Chunk (→{end}) already done "
                      f"(epoch={done}). Skipping.")
                continue
            print(f"ℹ️  Checkpoint at epoch {done}, "
                  f"resuming → {end}")
        else:
            print(f"ℹ️  Old-format checkpoint (epoch 30). "
                  f"Resuming from epoch 30 → {end}")

    rc = train_chunk(start, end, current_resume)

    if rc != 0:
        print(f"❌ Training failed. Return code: {rc}")
        break

    save_checkpoint_zip(chunk_end=end)
    current_resume = find_last_pth()

# ═══════════════════════════════════════════════════════════════
# STEP 8 — Final metrics
# ═══════════════════════════════════════════════════════════════
print("\n── Step 8: Final metrics ─────────────────────────────────")
logs = glob.glob(
    "/kaggle/working/LoMix/model_pth/**/log.txt", recursive=True)
logs = [l for l in logs if 'epo5' not in l]

if logs:
    with open(logs[0]) as f:
        lines = f.readlines()

    dice_data = []
    for line in lines:
        m = re.search(
            r'mean_dice\s*:\s*([\d.]+).*best_dice\s*:\s*([\d.]+)', line)
        if m:
            dice_data.append((float(m.group(1)), float(m.group(2))))

    if dice_data:
        best  = max(d[0] for d in dice_data)
        final = dice_data[-1][0]
        print("\n" + "═"*60)
        print(f"  FINAL RESULTS after {len(dice_data)} epochs")
        print(f"  Best  Dice : {best:.4f}")
        print(f"  Final Dice : {final:.4f}")
        print(f"  Paper Dice : 0.8260  (PVT-EMCAD + LoMix)")
        print(f"  Gap        : {0.8260 - best:.4f}")
        print("═"*60)
    else:
        print("No Dice scores found in log")
else:
    print("No log file found")

── Step 1: Cloning repo ──────────────────────────────────
✓ Repo cloned

── Step 2: Installing packages ───────────────────────────
✓ Packages installed

── Step 3: Restoring files from bundle ───────────────────
✓ Using bundle files directly from /kaggle/input/datasets/ajfaisal002/lomix-synapse-preprocessed/lomix_restart_bundle
  Patch dir: ['trainer.py', 'pvt_v2_b2.pth', 'train_synapse_lomix.py']
✓ Restored train_synapse_lomix.py
✓ Restored trainer.py
✓ PVT weight ready  (101.5 MB)

── Step 4: Applying patches ──────────────────────────────
✓ Fixed PVT_CASCADE import
✓ Added --resume argument
✓ Added resume block to trainer.py
✓ Updated epoch iterator
✓ All patches applied

── Step 5: Import check ──────────────────────────────────
✓ ptflops
✓ utils.utils
✓ trainer
✓ EMCADNet

── Step 7: Training ──────────────────────────────────────
   Chunks      : [(30, 150)]
   Resume from : /kaggle/input/datasets/ajfaisal002/lomix-synapse-preprocessed/lomix_ckpt_epoch30/last.pth
ℹ️  Old-format

  0%|                                         | 0/120 [00:00<?, ?it/s]

Original Weights (raw): %s -0.0164 -0.0153 -0.0122 -0.0142
Original Weights (softplus): %s 0.6850 0.6855 0.6871 0.6861
   => sum(original) = %.4f 2.743628978729248
Synthesized Weights for '%s' (raw): %s add -0.0166 -0.0146 -0.0113 -0.0161 -0.0131 -0.0117 -0.0159 -0.0140 -0.0132 -0.0143 -0.0139
Synthesized Weights for '%s' (softplus): %s add 0.6849 0.6859 0.6875 0.6851 0.6866 0.6873 0.6852 0.6862 0.6866 0.6860 0.6862
   => sum(%s) = %.4f add 7.547544002532959
Synthesized Weights for '%s' (raw): %s mul -0.0155 -0.0133 -0.0096 -0.0154 -0.0106 -0.0093 -0.0152 -0.0121 -0.0116 -0.0132 -0.0140
Synthesized Weights for '%s' (softplus): %s mul 0.6854 0.6865 0.6884 0.6855 0.6878 0.6885 0.6856 0.6871 0.6874 0.6866 0.6862
   => sum(%s) = %.4f mul 7.555002212524414
Synthesized Weights for '%s' (raw): %s wf -0.0152 -0.0127 -0.0135 -0.0129 -0.0129 -0.0133 -0.0135 -0.0131 -0.0122 -0.0121 -0.0131
Synthesized Weights for '%s' (softplus): %s wf 0.6856 0.6868 0.6864 0.6867 0.6867 0.6865 0.6864 0.6866 0.687


0it [00:00, ?it/s]
1it [00:11, 11.85s/it]
2it [00:17,  8.04s/it]
3it [00:23,  7.12s/it]
4it [00:33,  8.33s/it]
5it [00:41,  8.28s/it]
6it [00:49,  8.26s/it]
7it [00:56,  7.57s/it]
8it [01:07,  8.70s/it]
9it [01:15,  8.61s/it]
10it [01:23,  8.37s/it]
11it [01:28,  7.23s/it]
12it [01:32,  7.75s/it]
  1%|▎                             | 1/120 [04:41<9:18:01, 281.36s/it]

Original Weights (raw): %s -0.0351 -0.0329 -0.0256 -0.0194
Original Weights (softplus): %s 0.6758 0.6768 0.6804 0.6835
   => sum(original) = %.4f 2.716548204421997
Synthesized Weights for '%s' (raw): %s add -0.0336 -0.0296 -0.0237 -0.0314 -0.0265 -0.0242 -0.0308 -0.0266 -0.0253 -0.0267 -0.0271
Synthesized Weights for '%s' (softplus): %s add 0.6765 0.6785 0.6813 0.6776 0.6800 0.6811 0.6778 0.6799 0.6806 0.6799 0.6797
   => sum(%s) = %.4f add 7.472859859466553
Synthesized Weights for '%s' (raw): %s mul -0.0323 -0.0280 -0.0209 -0.0308 -0.0226 -0.0197 -0.0307 -0.0244 -0.0236 -0.0266 -0.0300
Synthesized Weights for '%s' (softplus): %s mul 0.6771 0.6792 0.6827 0.6779 0.6819 0.6834 0.6779 0.6810 0.6814 0.6799 0.6783
   => sum(%s) = %.4f mul 7.480813026428223
Synthesized Weights for '%s' (raw): %s wf -0.0327 -0.0268 -0.0219 -0.0270 -0.0210 -0.0198 -0.0286 -0.0246 -0.0223 -0.0220 -0.0240
Synthesized Weights for '%s' (softplus): %s wf 0.6770 0.6798 0.6822 0.6797 0.6827 0.6833 0.6789 0.6809 0.682

...

Original Weights (raw): %s -2.0523 -1.9594 -1.7587 -1.5568
Original Weights (softplus): %s 0.1208 0.1319 0.1589 0.1913
   => sum(original) = %.4f 0.6029189229011536
Synthesized Weights for '%s' (raw): %s add -1.8985 -1.6842 -1.5560 -1.6835 -1.5768 -1.5859 -1.8524 -1.7844 -1.7834 -1.7919 -1.8051
Synthesized Weights for '%s' (softplus): %s add 0.1396 0.1702 0.1914 0.1704 0.1878 0.1863 0.1457 0.1552 0.1553 0.1541 0.1523
   => sum(%s) = %.4f add 1.808373212814331
Synthesized Weights for '%s' (raw): %s mul -1.8864 -1.7108 -1.5957 -1.7320 -1.6266 -1.6225 -1.9087 -1.8470 -1.8478 -1.8850 -1.8849
Synthesized Weights for '%s' (softplus): %s mul 0.1412 0.1661 0.1846 0.1629 0.1795 0.1802 0.1383 0.1464 0.1463 0.1414 0.1414
   => sum(%s) = %.4f mul 1.7282068729400635
Synthesized Weights for '%s' (raw): %s wf -1.9587 -1.7631 -1.5956 -1.7623 -1.5928 -1.5778 -1.7651 -1.6157 -1.6069 -1.6049 -1.6127
Synthesized Weights for '%s' (softplus): %s wf 0.1319 0.1583 0.1846 0.1584 0.1851 0.1877 0.1580 0.1813 0.1


0it [00:00, ?it/s]
1it [00:11, 11.92s/it]
2it [00:17,  8.02s/it]
3it [00:23,  7.13s/it]
4it [00:33,  8.39s/it]
5it [00:42,  8.43s/it]
6it [00:50,  8.42s/it]
7it [00:56,  7.73s/it]
8it [01:07,  8.81s/it]
9it [01:16,  8.73s/it]
10it [01:24,  8.50s/it]
11it [01:29,  7.36s/it]
12it [01:34,  7.86s/it]
 99%|███████████████████████████▊| 119/120 [9:00:05<04:30, 270.75s/it]

Original Weights (raw): %s -2.0695 -1.9757 -1.7734 -1.5704
Original Weights (softplus): %s 0.1189 0.1299 0.1568 0.1889
   => sum(original) = %.4f 0.5944623351097107
Synthesized Weights for '%s' (raw): %s add -1.9145 -1.6984 -1.5691 -1.6977 -1.5906 -1.6003 -1.8696 -1.8014 -1.8008 -1.8093 -1.8222
Synthesized Weights for '%s' (softplus): %s add 0.1375 0.1680 0.1892 0.1681 0.1855 0.1839 0.1434 0.1528 0.1529 0.1517 0.1499
   => sum(%s) = %.4f add 1.78273606300354
Synthesized Weights for '%s' (raw): %s mul -1.9026 -1.7256 -1.6097 -1.7493 -1.6434 -1.6397 -1.9259 -1.8637 -1.8646 -1.9023 -1.9019
Synthesized Weights for '%s' (softplus): %s mul 0.1390 0.1639 0.1823 0.1603 0.1767 0.1773 0.1361 0.1442 0.1441 0.1391 0.1391
   => sum(%s) = %.4f mul 1.7021597623825073
Synthesized Weights for '%s' (raw): %s wf -1.9750 -1.7779 -1.6092 -1.7771 -1.6064 -1.5914 -1.7799 -1.6293 -1.6204 -1.6184 -1.6263
Synthesized Weights for '%s' (softplus): %s wf 0.1299 0.1561 0.1824 0.1563 0.1828 0.1854 0.1559 0.1790 0.18


0it [00:00, ?it/s]
1it [00:12, 12.16s/it]
2it [00:17,  8.20s/it]
3it [00:23,  7.21s/it]
4it [00:34,  8.54s/it]
5it [00:42,  8.53s/it]
6it [00:51,  8.46s/it]
7it [00:57,  7.77s/it]
8it [01:08,  8.92s/it]
9it [01:17,  8.80s/it]
10it [01:25,  8.49s/it]
11it [01:29,  7.32s/it]
12it [01:34,  7.90s/it]
 99%|███████████████████████████▊| 119/120 [9:04:40<04:34, 274.62s/it]



════════════════════════════════════════════════════════════
  ✅  COMPLETE — epochs up to 150
  📦  FILE NAME : lomix_ckpt_epoch150_20260817_2012.zip
      SIZE      : 198.9 MB
  ⬇️   DOWNLOAD  : Output tab → lomix_ckpt_epoch150_20260817_2012.zip
════════════════════════════════════════════════════════════


── Step 8: Final metrics ─────────────────────────────────
No log file found
